# Notebook 80: V8.9.1 Signed Mask-Offset Stress Test

Quick diagnostic rerunning the complete corrected V8.8 pipeline at authority-mask offsets **-2, -1, 0, +1 and +2** relative to the undilated reviewed seed. Negative offsets erode the seed; positive offsets dilate it. **+2 is the current V8.8 authority geometry and the comparator.** This is expected to fail visually and has no promotion authority.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn joblib torch torchvision pyarrow matplotlib


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json,hashlib,shutil,importlib.util,sys
import cv2,numpy as np,pandas as pd
import matplotlib.pyplot as plt
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
OPERATOR_SOURCE=ROOT/'v8_9_release'/'v8_9_1_operator.py'
P_PRINT_BUNDLE=ROOT/'observed_print_full_development_freeze_v8_2f'/'20261117_120424_v8_2f_full_development_p_print_freeze'/'p_print_provider'
REMOVE_BUNDLE=ROOT/'m5_remove_full_development_freeze_v8_8f'/'20261117_110512_notebook52f_v8_8_full_development_remove_freeze'/'full_development_remove'
NB72_ROOT=ROOT/'notebook72_frozen_v8_8_annotated_only_operational_trial'
OFFSETS=[-2,-1,0,1,2]
EXPECTED_PAGES=151
RUN=ROOT/'v8_9_1_mask_offset_stress_test'/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook80')
for p in [RUN,RUN/'pages',RUN/'tables',RUN/'visual',RUN/'audit',RUN/'upload']:p.mkdir(parents=True,exist_ok=False)
def require(x,m):
 if not bool(x):raise RuntimeError(m)
def sha256_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def latest(root,required):
 hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in required)]
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
require(OPERATOR_SOURCE.exists(),f'Missing {OPERATOR_SOURCE}')
spec=importlib.util.spec_from_file_location('v8_9_1_operator',OPERATOR_SOURCE);mod=importlib.util.module_from_spec(spec);sys.modules['v8_9_1_operator']=mod;spec.loader.exec_module(mod)
operators={o:mod.load_v891_operator(P_PRINT_BUNDLE,REMOVE_BUNDLE,mask_offset=o) for o in OFFSETS}
NB72=latest(NB72_ROOT,['inputs/runtime_roster.csv']);require(NB72 is not None,'Complete Notebook72 run not found.')
roster=pd.read_csv(NB72/'inputs/runtime_roster.csv');required={'page','observed_path','seed_mask_path'};require(required<=set(roster),f'Roster missing {sorted(required-set(roster))}');require(len(roster)==151,f'Expected 151 pages, found {len(roster)}')
print('Run:',RUN)


## One-page smoke test

In [ ]:
r=roster.iloc[0];page=str(r.page);obs=cv2.imread(str(r.observed_path),0);seed=cv2.imread(str(r.seed_mask_path),0);require(obs is not None and seed is not None and obs.shape==seed.shape,f'{page}: input failure')
smoke=[]
for o,op in operators.items():
 a=op.run_page(obs,seed,page=page);b=op.run_page(obs,seed,page=page)
 smoke.append({'offset':o,'authority_pixels':int(a.authoritative_mask.sum()),'changed_vs_observed':int(np.sum(a.corrected!=a.observed)),'outside_authority_drift':a.experimental_audit['outside_authority_drift'],'exact_replay':np.array_equal(a.corrected,b.corrected)})
smoke=pd.DataFrame(smoke);smoke.to_csv(RUN/'audit'/'smoke_test.csv',index=False);require(smoke.exact_replay.all() and (smoke.outside_authority_drift==0).all(),'Smoke test failed.');require(smoke.sort_values('offset').authority_pixels.is_monotonic_increasing,'Authority area is not monotonic with offset.');display(smoke)


## Run all five mask offsets on the 151 HITL pages

In [ ]:
rows=[]
for _,r in roster.iterrows():
 page=str(r.page);obs=cv2.imread(str(r.observed_path),0);seed=cv2.imread(str(r.seed_mask_path),0);require(obs is not None and seed is not None and obs.shape==seed.shape,f'{page}: input failure')
 d=RUN/'pages'/page;d.mkdir(parents=True,exist_ok=False);cv2.imwrite(str(d/'observed.png'),obs);cv2.imwrite(str(d/'reviewed_seed.png'),seed)
 outputs={}
 for o,op in operators.items():
  z=op.run_page(obs,seed,page=page);outputs[o]=z.corrected;tag=f'n{abs(o)}' if o<0 else f'p{o}'
  cv2.imwrite(str(d/f'offset_{tag}.png'),z.corrected);cv2.imwrite(str(d/f'authority_{tag}.png'),z.authoritative_mask.astype(np.uint8)*255)
  rows.append({'page':page,'offset':o,'seed_pixels':int((seed>0).sum()),'authority_pixels':int(z.authoritative_mask.sum()),'changed_vs_observed':int(np.sum(z.corrected!=obs)),'different_vs_p2':0,'outside_authority_drift':z.experimental_audit['outside_authority_drift']})
 for row in rows[-len(OFFSETS):]:row['different_vs_p2']=int(np.sum(outputs[row['offset']]!=outputs[2]))
 (d/'page_audit.json').write_text(json.dumps({'page':page,'offsets':{str(o):operators[o].run_page(obs,seed,page=page).experimental_audit for o in OFFSETS}},indent=2))
 print(page)
results=pd.DataFrame(rows);results.to_csv(RUN/'tables'/'mask_offset_results.csv',index=False);require(len(results)==151*5,'Incomplete results.');require((results.outside_authority_drift==0).all(),'Outside-authority drift detected.')


## Ranked six-column contact sheets: observed, -2, -1, 0, +1, +2

In [ ]:
wide=results.pivot(index='page',columns='offset',values='different_vs_p2').reset_index();wide['max_difference_vs_p2']=wide[[-2,-1,0,1]].max(axis=1);wide=wide.sort_values('max_difference_vs_p2',ascending=False);wide.to_csv(RUN/'tables'/'pages_ranked_by_mask_offset_effect.csv',index=False)
contact=[];PER=6
for start in range(0,len(wide),PER):
 q=wide.iloc[start:start+PER];fig,axes=plt.subplots(len(q),6,figsize=(24,max(5,5*len(q))),squeeze=False)
 for rr,(_,row) in enumerate(q.iterrows()):
  d=RUN/'pages'/str(row.page);paths=[d/'observed.png',d/'offset_n2.png',d/'offset_n1.png',d/'offset_p0.png',d/'offset_p1.png',d/'offset_p2.png'];titles=['Observed','-2 erosion','-1 erosion','0 seed','+1 dilation','+2 current']
  for cc,(p,t) in enumerate(zip(paths,titles)):
   im=cv2.imread(str(p),0);axes[rr,cc].imshow(im,cmap='gray',vmin=0,vmax=255);axes[rr,cc].set_title((str(row.page)+'\n'+t) if cc==0 else t,fontsize=8);axes[rr,cc].axis('off')
 plt.tight_layout();p=RUN/'visual'/f'contact_sheet_{start//PER+1:02d}.png';plt.savefig(p,dpi=105,bbox_inches='tight');plt.close(fig);contact.append(p)
print('Contact sheets:',len(contact))


## Diagnostic summary and upload bundle

In [ ]:
summary_by_offset=results.groupby('offset',as_index=False).agg(pages=('page','nunique'),authority_pixels=('authority_pixels','sum'),changed_vs_observed=('changed_vs_observed','sum'),different_vs_p2=('different_vs_p2','sum'),outside_authority_drift=('outside_authority_drift','sum'));summary_by_offset.to_csv(RUN/'tables'/'summary_by_offset.csv',index=False)
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.9.1-mask-offset-stress-test','status':'DIAGNOSTIC_ONLY_EXPECTED_TO_FAIL','pages':151,'offsets':OFFSETS,'offset_definition':{'-2':'5x5 elliptical erosion of reviewed seed','-1':'3x3 elliptical erosion of reviewed seed','0':'reviewed seed','1':'3x3 elliptical dilation of reviewed seed','2':'5x5 elliptical dilation of reviewed seed; current V8.8 authority'},'complete_pipeline_rerun_for_each_offset':True,'uses_clean_truth':False,'promotion_authority':False,'operator_sha256':sha256_file(OPERATOR_SOURCE)};(RUN/'diagnostic_summary.json').write_text(json.dumps(summary,indent=2))
for p in [RUN/'diagnostic_summary.json',RUN/'tables'/'summary_by_offset.csv',RUN/'tables'/'pages_ranked_by_mask_offset_effect.csv',RUN/'audit'/'smoke_test.csv',*contact]:shutil.copy2(p,RUN/'upload'/p.name)
shutil.copy2(OPERATOR_SOURCE,RUN/'upload'/'v8_9_1_operator.py');shutil.make_archive(str(RUN/'notebook80_upload_bundle'),'zip',RUN/'upload');display(summary_by_offset);print(json.dumps(summary,indent=2));print('COMPLETE:',RUN)
